# H&M — Test 4 Cleaned Datasets x 3 Algorithms (GPU)

**Loads parquet files** from the `clean dataset/` folder (created by cleaning_versions.ipynb).

**4 datasets:**
1. normal_clean — NaNs filled with fixed values
2. mean_version — numeric NaNs filled with mean, categorical with mode
3. mode_version — all NaNs filled with mode
4. median_version — numeric NaNs filled with median, categorical with mode

**3 algorithms:**
- LightGBM (tries GPU, falls back to CPU)
- TensorFlow Neural Network (DirectML for AMD RX 6700 XT)
- RandomForest (CPU only)

**Task:** Binary classification — will customer X buy article Y?
- Features = customer info + article info (10 features, NO price/channel to avoid leakage)
- Metric = accuracy

**Environment:** Run `conda activate Heavy` before launching.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# --- GPU Setup for TensorFlow (DirectML for AMD) ---
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices(gpus[0], 'GPU')
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'TensorFlow GPU detected: {gpus[0]}')
else:
    print('No GPU detected for TensorFlow — will use CPU')

print(f'TensorFlow version: {tf.__version__}')

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print('All imports ready.')

TensorFlow GPU detected: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
TensorFlow version: 2.10.0
All imports ready.


## 1. Load parquet files from clean dataset folder

Parquet loads faster than CSV and keeps data types. We sample 500K transactions for speed.

In [2]:
CLEAN_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\clean dataset'

VERSION_NAMES = ['normal_clean', 'mean_version', 'mode_version', 'median_version']
VERSIONS = {}

for vname in VERSION_NAMES:
    vdir = os.path.join(CLEAN_DIR, vname)
    print(f'Loading {vname}...')
    
    a = pd.read_parquet(os.path.join(vdir, 'articles.parquet'))
    c = pd.read_parquet(os.path.join(vdir, 'customers.parquet'))
    t = pd.read_parquet(os.path.join(vdir, 'transactions.parquet'))
    
    # Sample 500K transactions for speed
    if len(t) > 500000:
        t = t.sample(n=500000, random_state=42)
    
    # Ensure IDs are strings
    a['article_id'] = a['article_id'].astype(str)
    c['customer_id'] = c['customer_id'].astype(str)
    t['article_id'] = t['article_id'].astype(str)
    t['customer_id'] = t['customer_id'].astype(str)
    
    VERSIONS[vname] = (a, c, t)
    print(f'  articles: {a.shape}, customers: {c.shape}, transactions: {t.shape}')

print(f'\nAll 4 versions loaded from parquet.')

Loading normal_clean...
  articles: (105542, 25), customers: (1371980, 7), transactions: (500000, 5)
Loading mean_version...
  articles: (105542, 25), customers: (1371980, 7), transactions: (500000, 5)
Loading mode_version...
  articles: (105542, 25), customers: (1371980, 7), transactions: (500000, 5)
Loading median_version...
  articles: (105542, 25), customers: (1371980, 7), transactions: (500000, 5)

All 4 versions loaded from parquet.


## 2. Prepare features for classification

NOTE: price and sales_channel_id are NOT used as features.
They leak the label — real purchases have price>0, fake negatives have price=0.
The model would just learn 'price>0 means bought' and get 100% accuracy.

Features used (10 total):
- Customer: age, FN, Active, club_member_status, fashion_news_frequency
- Article: product_type_no, colour_group_code, department_no, index_group_no, garment_group_no

In [3]:
CATEGORICAL_COLS = ['club_member_status', 'fashion_news_frequency']
NUMERIC_COLS = ['age', 'FN', 'Active',
                'product_type_no', 'colour_group_code', 'department_no',
                'index_group_no', 'garment_group_no']

def prepare_dataset(articles_df, customers_df, transactions_df, n_samples=100000):
    """Build a balanced binary classification dataset from a version."""
    # Sample positive examples (real purchases)
    pos = transactions_df.sample(n=min(n_samples, len(transactions_df)), random_state=42)
    pos = pos[['customer_id', 'article_id']].copy()
    pos['label'] = 1

    # Generate negative examples: same customers, random articles they didn't buy
    all_article_ids = articles_df['article_id'].unique()
    customer_bought = transactions_df.groupby('customer_id')['article_id'].apply(set).to_dict()

    neg_rows = []
    neg_customers = np.random.choice(pos['customer_id'].unique(), size=len(pos), replace=True)
    for cid in neg_customers:
        bought = customer_bought.get(cid, set())
        rand_article = str(np.random.choice(all_article_ids))
        tries = 0
        while rand_article in bought and tries < 10:
            rand_article = str(np.random.choice(all_article_ids))
            tries += 1
        neg_rows.append({'customer_id': cid, 'article_id': rand_article, 'label': 0})

    neg = pd.DataFrame(neg_rows)
    data = pd.concat([pos, neg], ignore_index=True)

    # Merge customer features (NO price, NO channel)
    cust_features = customers_df[['customer_id', 'age', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency']]
    data = data.merge(cust_features, on='customer_id', how='left')

    # Merge article features
    art_features = articles_df[['article_id', 'product_type_no', 'colour_group_code',
                                 'department_no', 'index_group_no', 'garment_group_no']]
    data = data.merge(art_features, on='article_id', how='left')

    # Fill any NaNs from merge
    for col in NUMERIC_COLS:
        if col in data.columns:
            data[col] = data[col].fillna(data[col].median() if data[col].notna().any() else 0)
    for col in CATEGORICAL_COLS:
        if col in data.columns:
            data[col] = data[col].fillna('UNKNOWN')

    # Encode categorical columns
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col].astype(str))

    # Final feature set (10 features)
    feature_cols = NUMERIC_COLS + CATEGORICAL_COLS
    X = data[feature_cols].values
    y = data['label'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    return X_train, X_test, y_train, y_test, feature_cols

print('Prepare function ready.')

Prepare function ready.


## 3. Training functions — LightGBM, TensorFlow, RandomForest

In [4]:
def train_lightgbm(X_train, y_train, X_test, y_test):
    """LightGBM — try GPU first, fallback to CPU."""
    try:
        model = lgb.LGBMClassifier(
            n_estimators=100, max_depth=10, learning_rate=0.1,
            device='gpu', random_state=42, verbose=-1
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        return accuracy_score(y_test, preds), 'GPU'
    except Exception as e:
        print(f'  LightGBM GPU failed ({str(e)[:60]}...), using CPU')
        model = lgb.LGBMClassifier(
            n_estimators=100, max_depth=10, learning_rate=0.1,
            random_state=42, verbose=-1
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        return accuracy_score(y_test, preds), 'CPU'


def train_tensorflow(X_train, y_train, X_test, y_test):
    """TensorFlow Neural Network — runs on GPU via DirectML."""
    input_dim = X_train.shape[1]
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=20, batch_size=512, verbose=0, validation_split=0.1)
    _, acc = model.evaluate(X_test, y_test, verbose=0)
    return acc, 'GPU' if gpus else 'CPU'


def train_randomforest(X_train, y_train, X_test, y_test):
    """RandomForest — CPU only."""
    model = RandomForestClassifier(n_estimators=100, max_depth=15,
                                     random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds), 'CPU'

print('Training functions ready.')

Training functions ready.


## 4. Run all 4 datasets x 3 algorithms

In [5]:
results = []

for version_name, (a, c, t) in VERSIONS.items():
    print(f'\n{"="*60}')
    print(f'  DATASET: {version_name}')
    print(f'{"="*60}')

    print('  Preparing features...')
    X_train, X_test, y_train, y_test, feat_cols = prepare_dataset(a, c, t, n_samples=100000)
    print(f'  Train: {X_train.shape}, Test: {X_test.shape}')
    print(f'  Features ({len(feat_cols)}): {feat_cols}')

    print('  Training LightGBM...')
    lgb_acc, lgb_dev = train_lightgbm(X_train, y_train, X_test, y_test)
    print(f'  LightGBM accuracy: {lgb_acc:.4f} ({lgb_dev})')

    print('  Training TensorFlow NN...')
    tf_acc, tf_dev = train_tensorflow(X_train, y_train, X_test, y_test)
    print(f'  TensorFlow accuracy: {tf_acc:.4f} ({tf_dev})')

    print('  Training RandomForest...')
    rf_acc, rf_dev = train_randomforest(X_train, y_train, X_test, y_test)
    print(f'  RandomForest accuracy: {rf_acc:.4f} ({rf_dev})')

    results.append({
        'dataset': version_name,
        'LightGBM': lgb_acc, 'LightGBM_device': lgb_dev,
        'TensorFlow': tf_acc, 'TensorFlow_device': tf_dev,
        'RandomForest': rf_acc, 'RandomForest_device': rf_dev,
    })

print(f'\n{"="*60}')
print('All 12 runs complete.')
print(f'{"="*60}')


  DATASET: normal_clean
  Preparing features...
  Train: (160000, 10), Test: (40000, 10)
  Features (10): ['age', 'FN', 'Active', 'product_type_no', 'colour_group_code', 'department_no', 'index_group_no', 'garment_group_no', 'club_member_status', 'fashion_news_frequency']
  Training LightGBM...
  LightGBM accuracy: 0.7262 (GPU)
  Training TensorFlow NN...
  TensorFlow accuracy: 0.6563 (GPU)
  Training RandomForest...
  RandomForest accuracy: 0.7265 (CPU)

  DATASET: mean_version
  Preparing features...
  Train: (160000, 10), Test: (40000, 10)
  Features (10): ['age', 'FN', 'Active', 'product_type_no', 'colour_group_code', 'department_no', 'index_group_no', 'garment_group_no', 'club_member_status', 'fashion_news_frequency']
  Training LightGBM...
  LightGBM accuracy: 0.7259 (GPU)
  Training TensorFlow NN...
  TensorFlow accuracy: 0.6208 (GPU)
  Training RandomForest...
  RandomForest accuracy: 0.7268 (CPU)

  DATASET: mode_version
  Preparing features...
  Train: (160000, 10), Test: (4

## 5. Results — per dataset table

In [6]:
for r in results:
    ds = r['dataset']
    models = {'LightGBM': r['LightGBM'], 'TensorFlow': r['TensorFlow'], 'RandomForest': r['RandomForest']}
    best_model = max(models, key=models.get)
    best_acc = models[best_model]

    print(f'\n{"="*50}')
    print(f'  Dataset: {ds}')
    print(f'{"="*50}')
    print(f'{"Model":<18} {"Accuracy":<12} {"Device":<8}  {"Best?"}')
    print('-' * 50)
    for model_name, acc in models.items():
        dev = r[f'{model_name}_device']
        is_best = '<-- BEST' if model_name == best_model else ''
        print(f'{model_name:<18} {acc:<12.4f} {dev:<8}  {is_best}')
    print(f'\n  Highest accuracy: {best_model} ({best_acc:.4f})')


  Dataset: normal_clean
Model              Accuracy     Device    Best?
--------------------------------------------------
LightGBM           0.7262       GPU       
TensorFlow         0.6563       GPU       
RandomForest       0.7265       CPU       <-- BEST

  Highest accuracy: RandomForest (0.7265)

  Dataset: mean_version
Model              Accuracy     Device    Best?
--------------------------------------------------
LightGBM           0.7259       GPU       
TensorFlow         0.6208       GPU       
RandomForest       0.7268       CPU       <-- BEST

  Highest accuracy: RandomForest (0.7268)

  Dataset: mode_version
Model              Accuracy     Device    Best?
--------------------------------------------------
LightGBM           0.7265       GPU       <-- BEST
TensorFlow         0.6611       GPU       
RandomForest       0.7256       CPU       

  Highest accuracy: LightGBM (0.7265)

  Dataset: median_version
Model              Accuracy     Device    Best?
-----------------

## 6. Final comparison — best model per dataset

In [7]:
print(f'{"="*65}')
print(f'  FINAL RESULTS — Best Model Per Dataset')
print(f'{"="*65}')
print(f'{"Dataset":<18} {"Best Model":<18} {"Accuracy":<12}')
print('-' * 65)

for r in results:
    ds = r['dataset']
    models = {'LightGBM': r['LightGBM'], 'TensorFlow': r['TensorFlow'], 'RandomForest': r['RandomForest']}
    best_model = max(models, key=models.get)
    best_acc = models[best_model]
    print(f'{ds:<18} {best_model:<18} {best_acc:<12.4f}')

all_best = [(r['dataset'], max(['LightGBM','TensorFlow','RandomForest'],
           key=lambda m: r[m]), max(r['LightGBM'], r['TensorFlow'], r['RandomForest'])) for r in results]
overall_ds, overall_model, overall_acc = max(all_best, key=lambda x: x[2])
print(f'\n  OVERALL WINNER: {overall_model} on {overall_ds} with {overall_acc:.4f} accuracy')

  FINAL RESULTS — Best Model Per Dataset
Dataset            Best Model         Accuracy    
-----------------------------------------------------------------
normal_clean       RandomForest       0.7265      
mean_version       RandomForest       0.7268      
mode_version       LightGBM           0.7265      
median_version     RandomForest       0.7290      

  OVERALL WINNER: RandomForest on median_version with 0.7290 accuracy


In [8]:
results_df = pd.DataFrame(results)
results_df = results_df[['dataset', 'LightGBM', 'LightGBM_device',
                         'TensorFlow', 'TensorFlow_device',
                         'RandomForest', 'RandomForest_device']]
results_df.columns = ['Dataset', 'LightGBM', 'LGBM_Device',
                      'TensorFlow', 'TF_Device',
                      'RandomForest', 'RF_Device']
results_df

,Dataset,LightGBM,LGBM_Device,TensorFlow,TF_Device,RandomForest,RF_Device
0,normal_clean,0.726200,GPU,0.656300,GPU,0.72645,CPU
1,mean_version,0.725925,GPU,0.620800,GPU,0.72675,CPU
2,mode_version,0.726525,GPU,0.661075,GPU,0.72555,CPU
3,median_version,0.727675,GPU,0.652050,GPU,0.72900,CPU


## Summary

| What | Details |
|------|---------|
| Data format | Parquet (from clean dataset/ folder) |
| Datasets tested | 4 (normal_clean, mean_version, mode_version, median_version) |
| Algorithms | 3 (LightGBM, TensorFlow NN, RandomForest) |
| Total runs | 12 |
| Task | Binary classification: will customer buy article? |
| Metric | Accuracy |
| Sample size | 100K positive + 100K negative = 200K per dataset |
| Features | 10 (age, FN, Active, club status, news freq, product type, color, dept, index group, garment group) |
| GPU | TensorFlow via DirectML, LightGBM tries GPU then CPU, RandomForest CPU only |